# Signal Monitor Notebook

Runs a short lock-in signal monitor measurement and shows a live notebook plot.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src" / "kohdalab").exists():
            return path
    raise RuntimeError("Could not find repository root containing src/kohdalab.")


repo_root = find_repo_root()
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
config_path = repo_root / "src" / "kohdalab" / "config" / "trkr_config_kikuchi.json"

from kohdalab.api import (
    Experiment,
    format_point,
    load_config,
    make_signal_monitor_live_update,
    signal_monitor_plan,
)

config = load_config(config_path)
experiment = Experiment(config)

def print_status(status: str) -> None:
    print(f"status: {status}", flush=True)


In [ ]:
interval_s = 1.0
n_points = 10
output = None  # None uses the output path configured for signal_monitor.
y_key = "R_V"

plan = signal_monitor_plan(interval_s=interval_s, n_points=n_points)
live_update = make_signal_monitor_live_update(y_key=y_key, title="Signal Monitor")
axis_key = "elapsed_s"


In [ ]:
def handle_point(point):
    print(format_point(point, axis_key=axis_key), flush=True)
    live_update(point)


rows = experiment.run_signal_monitor(
    plan=plan,
    output=output,
    on_status=print_status,
    on_point=handle_point,
)

display(pd.DataFrame(rows))


In [ ]:
# Run this when you are done with the notebook session.
# experiment.disconnect_all()
